# L13c: The Model Context Protocol
In this lecture, we introduce the Model Context Protocol (MCP), an open standard that connects language model applications to tools. We look at the roles in an MCP system, the messages of a tool call, and the checks that keep a tool call safe.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe the roles in an MCP system:__ A host runs a language model and connects to servers, which offer tools. We'll define the host, the client, and the server.
> * __Read the messages of a tool call:__ A client and a server exchange JSON-RPC messages. We'll follow the messages that discover a server, list its tools, and call a tool.
> * __Identify the trust boundaries of a tool call:__ A tool runs code on the user's computer, and the language model writes its arguments. We'll look at what the host and the server each check.

Let's get started!

___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Build an MCP server for the National Weather Service API](CHEME-5800-L13c-Example-MCP-NWS-Fall-2026.ipynb). In this example, we wrap our software development kit (SDK) for the National Weather Service API in an MCP server with two tools. We call the tools as a host would, and trigger both kinds of errors.

___

## Language Models and Tools
A __large language model (LLM)__ is a neural network that reads and writes text. On its own, an LLM cannot look up today's weather or run our Julia code. An application gives an LLM these abilities through __tools__, functions that the application runs when the model asks for them.

In a __tool call__, the model answers a question with the name of a tool and its arguments. The application runs the tool, and the model writes its answer from the result.

The __Model Context Protocol (MCP)__ is an open standard that connects applications to tools. Without a standard, $M$ applications and $N$ tool sources need up to $M\times{N}$ custom integrations. With MCP, each side implements the protocol once, for $M+N$ implementations.

Anthropic released MCP in November 2024, and donated it to the Agentic AI Foundation of the Linux Foundation in December 2025. We use [version `2026-07-28` of the MCP specification](https://modelcontextprotocol.io/specification/2026-07-28).

___

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>

## Hosts, Clients, and Servers
An MCP system has three roles:

<p align="center">
    <img class="course-diagram" src="figs/Fig-MCP-Architecture/Fig-MCP-Architecture.svg" width="1100" style="max-width:100%; height:auto;" alt="A User box on the left is joined by a two-way arrow to a large blue box labeled Host. Inside the host, a Language model box is joined by gray lines to two Client boxes. A blue two-way arrow labeled MCP messages joins the upper client to a Weather server box, and another joins the lower client to a FRED server box. A dark two-way arrow labeled HTTPS joins the weather server to an NWS API box, and another joins the FRED server to a FRED API box. Notes below say that the host is a chat app, a coding assistant, or our notebook; the servers are our Julia code, started by the host; and the APIs are RESTful web services."/>
</p>

* __Host:__ The host is the application the user works with, such as a chat app or coding assistant. It manages the model, starts our servers, and controls tool calls.
* __Client:__ Each client runs inside the host and exchanges MCP messages with one server.
* __Server:__ A server provides tools. Our Julia servers call the NWS and FRED APIs over HTTPS.

A __transport__ carries messages between a client and a server. Our servers use __stdio__: the host sends requests to the server's __standard input__ and reads replies from its __standard output__.

The protocol also supports __Streamable HTTP__ for servers accessed over a network.

___

## MCP Messages
The protocol uses __JSON-RPC 2.0__, a JSON format for requests and responses:

* __Request:__ A request carries `jsonrpc` (always `"2.0"`), an `id`, the `method`, and its `params`.
* __Response:__ A response repeats the request's `id` and contains either a `result` or an `error`.
* __Notification:__ A notification has no `id` and receives no response.

With __stdio__, each message is one line of JSON text. Standard output carries only protocol messages. Julia sends log messages to __standard error__.

Version `2026-07-28` is __stateless__: every request carries its protocol version, client identity, and capabilities in `params._meta`. The server checks the version on every request. Our servers reject the older `initialize` handshake with error `-32022` because it lacks this metadata.

An `error` contains a numeric `code` and a `message`. Our servers use six codes:

| Code | Meaning |
| --- | --- |
| `-32700` | Parse error: the message is not valid JSON |
| `-32600` | Invalid request: the message is not a JSON-RPC 2.0 request, e.g., its `jsonrpc` field is missing |
| `-32601` | Method not found: the server has no method with this name |
| `-32602` | Invalid params, e.g., the request names a tool that the server does not have |
| `-32603` | Internal error: the server failed while it processed the request |
| `-32022` | Unsupported protocol version |

### Discover a Server
The [messages folder](messages) contains our saved weather-server requests and responses.

A `server/discover` request asks which protocol versions and capabilities the server supports.

__What do we see?__ Our server advertises tools and supplies `instructions` for the model. Its identity appears in `_meta`. The `ttlMs` value permits reuse of this response for one hour.

__Messages:__ [Request](messages/1-discover-request.json) · [Response](messages/1-discover-response.json).

### List the Tools
A `tools/list` request returns each tool's `name`, `description`, and `inputSchema`. The model uses the description to choose a tool.

The input schema uses __JSON Schema__ to define arguments, their types, and which are required. For `get_hourly_forecast`, `latitude` and `longitude` are required numbers. The optional `hours` argument is an integer from `1` to `48`.

__Messages:__ [Request](messages/2-list-request.json) · [Response](messages/2-list-response.json).

### Call a Tool
A `tools/call` request gives the tool's name and arguments. Here, we request a one-hour forecast for Ithaca, NY.

__What do we see?__ The `content` array provides text for the model. The `structuredContent` field provides the same result as JSON data for a program. The `isError` flag is `false`, indicating that the tool completed without an error.

__Messages:__ [Request](messages/3-call-request.json) · [Response](messages/3-call-response.json).

The figure follows the call from the user's question to the model's answer:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>

<p align="center">
    <img class="course-diagram" src="figs/Fig-MCP-ToolCall/Fig-MCP-ToolCall.svg" width="1000" style="max-width:100%; height:auto;" alt="A sequence diagram with four dashed lifelines: Language model, Host, MCP server, and Web API. The user asks the host: Will it rain in Ithaca tonight? Step 1, a gray arrow from the host to the model: question and tool list. Step 2, a gray arrow from the model to the host: tool name and arguments. Step 3, a blue arrow from the host to the MCP server: tools/call request. Step 4, a dark arrow from the server to the web API: HTTPS GET. Step 5, a dark arrow back to the server: JSON forecast. Step 6, a blue arrow from the server to the host: tools/call result. Step 7, a gray arrow from the host to the model: tool result. Step 8, a gray arrow from the model to the host: answer. The host shows the answer to the user."/>
</p>

__One tool call.__ Gray arrows stay within the host, blue arrows carry MCP messages, and dark arrows carry HTTPS requests and responses. Our weather server makes two API requests.

### Two Kinds of Errors
A __protocol error__ means the request cannot be processed. Our request names an unavailable `get_snowfall` tool. The response contains an `error` with code `-32602` and no `result`.

__Messages:__ [Request](messages/4-unknown-tool-request.json) · [Response](messages/4-unknown-tool-response.json).

A __tool execution error__ means the tool ran but failed. Our Paris forecast request reaches the NWS API, which does not cover that location. The response contains a `result` with `isError` set to `true`.

__Messages:__ [Request](messages/5-paris-request.json) · [Response](messages/5-paris-response.json).

Protocol errors usually indicate a host or server bug. Tool execution errors return an explanation that the model can use to choose different arguments.

### The Server Loop
Our servers run one loop, which reads one message and writes at most one response:

__Initialize__: Given a set of tools $\mathcal{T}$, each with a name, a description, an input schema, and a handler (the function that runs the tool), and the protocol version $v\gets$ `2026-07-28`.

While the standard input is open __do__:
1. Read the next line $\ell$ that is not blank, and parse it: $m\gets\texttt{parse}(\ell)$.
2. Choose the response $r$:
    - If $\ell$ is not valid JSON __then__ $r\gets$ a parse error, `-32700`.
    - Otherwise, if $m$ is not a JSON-RPC 2.0 message __then__ $r\gets$ an invalid-request error, `-32600`.
    - Otherwise, if $m$ has no `id` __then__ $r\gets\texttt{nothing}$ (notification).
    - Otherwise, if the `_meta` object of $m$ does not name $v$ __then__ $r\gets$ an unsupported-version error, `-32022`.
    - Otherwise, if the method of $m$ is `server/discover` __then__ $r\gets$ the versions, capabilities, and identity of the server.
    - Otherwise, if the method is `tools/list` __then__ $r\gets$ the name, description, and input schema of each tool in $\mathcal{T}$.
    - Otherwise, if the method is `tools/call` and $\mathcal{T}$ holds the named tool __then__ run its handler, and set $r\gets$ its result, with `isError` set to `true` if the handler fails (tool execution error).
    - Otherwise, if the method is `tools/call` __then__ $r\gets$ an invalid-params error, `-32602` (unknown tool).
    - Otherwise, $r\gets$ a method-not-found error, `-32601`.
3. If $r\neq\texttt{nothing}$ __then__ write $r$ as one line of JSON text to the standard output.

__Output__: None. The loop ends when the client closes the standard input of the server.

[The `process_message(...)` function](src/Server.jl) carries out step 2 for one line, and [the `serve(...)` function](src/Server.jl) runs the loop.

___

## Trust and Security
The weather example crosses three __trust boundaries__:

* __Model → host:__ The model proposes a tool and arguments. The host decides whether the call is permitted and asks for user approval when needed.
* __Host → server:__ Our stdio servers run locally with the user's permissions. Install only servers whose code you trust.
* __Server → API:__ The server validates tool arguments before making an API request. It keeps API keys out of MCP messages.

__Prompt injection__ occurs when text returned by a server tries to redirect the model toward actions the user did not request.

Our tools read public data. Tools that change data, such as placing an order, should require user approval for each call.

___

## Summary
In this lecture, we introduced the Model Context Protocol, the messages of a tool call, and the checks that keep a tool call safe.

> __Key Takeaways:__
>
> * __Hosts, clients, and servers:__ A host runs a language model and one client for each MCP server, and each server offers tools. The model never talks to a server, because every tool call passes through the host.
> * __Discover, list, and call:__ A client discovers a server, lists its tools, and calls a tool with JSON-RPC requests that name the protocol version. A wrong request gets a protocol error, while a tool that fails returns a result with its error flag set.
> * __Trust boundaries:__ A server runs code with the permissions of the user, and the language model writes the arguments of each call. The host approves each call, and the server checks every argument and keeps API keys to itself.

With MCP, any language model application that supports this version of the protocol can use our own computational code.

___